# Batch DenoIST over a folder of Zarr files

This notebook runs `denoistpy` over every input `.zarr` store in a folder, writes corrected outputs, and creates QC tables/plots for each sample plus a combined batch summary.

It is configured for Proseg-style `SpatialData` Zarr stores by default, with an AnnData table at `tables["table"]` and transcript points at `points["transcripts"]`.

## 1. Setup

Install the package in editable mode from the repository root before running the notebook if needed:

```bash
python -m pip install --editable ".[spatialdata]"
```

In [7]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spatialdata as sd

from denoistpy import (
    denoist,
    from_proseg_spatialdata,
    summarize_denoist_result,
    write_report_csv,
    write_result_anndata_zarr,
    write_result_h5ad,
    write_result_spatialdata_zarr,
)

## 2. Parameters

Edit these paths and options for your data. If you use Papermill, this cell can be parameterized.

In [8]:
# Input/output folders
INPUT_DIR = Path("E:/cosmx/proseg_out")
OUTPUT_DIR = Path("E:/cosmx/denoistpy_gh_out")

# Zarr discovery
INPUT_GLOB = "*.zarr"
RECURSIVE = False
SAMPLE_LIMIT = None  # set to an integer for a smoke test

# SpatialData/Proseg layout
TABLE_KEY = "table"
POINTS_KEY = "transcripts"
LAYER = None  # None uses table.X
SPATIAL_KEY = "spatial"
CELL_COORDINATE_SOURCE = "obsm"  # "obsm" or "obs"
CELL_X_COL = None
CELL_Y_COL = None
GENE_NAMES_COL = None

# Transcript columns
TRANSCRIPT_POSITION = "adjusted"  # "adjusted" uses x/y, "observed" uses observed_x/observed_y
X_COL = None  # explicit override, or None for TRANSCRIPT_POSITION default
Y_COL = None
GENE_COL = "gene"
QV_COL = None  # Proseg usually has no qv column; use e.g. "qv" for Xenium-style points
QV_THRESHOLD = 20.0
BACKGROUND_COL = "background"
BACKGROUND_FILTER = None  # None, "exclude", or "only"

# DenoIST model options
DISTANCE = 50.0
NBINS = 200
POSTERIOR_CUTOFF = 0.6
N_INITS = 10
MAX_ITER = 5000
TOL = 1e-6
BACKEND = "torch"  # "torch" can be faster when PyTorch is installed and configured
DEVICE = "auto"
BATCH_SIZE = 1024
BACKGROUND_ONLY = False
INCLUDE_SELF_TWICE = False
STORE_MEMBERSHIPS = False  # dense genes x cells; enable mainly for small/debug runs
STORE_POSTERIOR = False     # dense genes x cells; enable mainly for small/debug runs
RANDOM_STATE = 0
DENOIST_PROGRESS = "auto"  # "auto" uses tqdm in notebooks and text elsewhere; use "none" to silence

# Outputs
OUTPUT_FORMAT = "spatialdata-zarr"  # "spatialdata-zarr", "h5ad", or "anndata-zarr"
OUTPUT_TABLE_KEY = "denoist"
COPY_UNS_FROM_TEMPLATE = False
OVERWRITE = False
TOP_N = 25
CONTINUE_ON_ERROR = True

## 3. Helpers

In [9]:
def discover_zarrs(input_dir: Path, pattern: str = "*.zarr", recursive: bool = False) -> list[Path]:
    paths = sorted(input_dir.rglob(pattern) if recursive else input_dir.glob(pattern))
    return [p for p in paths if p.is_dir()]


def clean_name(path: Path) -> str:
    return path.name.removesuffix(".zarr")


def prepare_dir(path: Path, overwrite: bool = False) -> Path:
    if path.exists():
        if not overwrite:
            raise FileExistsError(f"Output already exists: {path}")
        shutil.rmtree(path) if path.is_dir() else path.unlink()
    path.mkdir(parents=True, exist_ok=True)
    return path


def report_metric(report: dict[str, pd.DataFrame], metric: str, default=np.nan):
    summary = report["summary"]
    values = summary.loc[summary["metric"] == metric, "value"]
    return values.iloc[0] if len(values) else default


def plot_qc(report: dict[str, pd.DataFrame], sample_name: str, qc_dir: Path) -> list[Path]:
    qc_dir.mkdir(parents=True, exist_ok=True)
    written = []

    per_cell = report["per_cell"].copy()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(per_cell["removed_fraction"].fillna(0), bins=50, color="#4C78A8", edgecolor="white")
    ax.set_title(f"{sample_name}: removed fraction per cell")
    ax.set_xlabel("Removed fraction")
    ax.set_ylabel("Cells")
    fig.tight_layout()
    path = qc_dir / "removed_fraction_per_cell.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    written.append(path)

    top_genes = report["top_removed_genes"].sort_values("removed_counts")
    if not top_genes.empty:
        height = max(4, min(10, 0.32 * len(top_genes)))
        fig, ax = plt.subplots(figsize=(8, height))
        ax.barh(top_genes["gene"].astype(str), top_genes["removed_counts"], color="#F58518")
        ax.set_title(f"{sample_name}: top removed genes")
        ax.set_xlabel("Removed counts")
        ax.set_ylabel("")
        fig.tight_layout()
        path = qc_dir / "top_removed_genes.png"
        fig.savefig(path, dpi=150)
        plt.close(fig)
        written.append(path)

    status = report["model_status"].copy()
    if not status.empty:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(status["status"].astype(str), status["n_cells"], color="#54A24B")
        ax.set_title(f"{sample_name}: model status")
        ax.set_xlabel("Status")
        ax.set_ylabel("Cells")
        ax.tick_params(axis="x", labelrotation=30)
        fig.tight_layout()
        path = qc_dir / "model_status.png"
        fig.savefig(path, dpi=150)
        plt.close(fig)
        written.append(path)

    return written

## 4. Run One Sample

In [10]:
def run_one_zarr(zarr_path: Path) -> dict:
    sample_name = clean_name(zarr_path)
    sample_dir = prepare_dir(OUTPUT_DIR / sample_name, overwrite=OVERWRITE)
    qc_dir = sample_dir / "qc"
    report_dir = sample_dir / "report_csv"

    print(f"Reading {zarr_path}")
    sdata = sd.read_zarr(zarr_path)
    inp = from_proseg_spatialdata(
        sdata,
        table_key=TABLE_KEY,
        points_key=POINTS_KEY,
        layer=LAYER,
        transcript_position=TRANSCRIPT_POSITION,
        x_col=X_COL,
        y_col=Y_COL,
        gene_col=GENE_COL,
        background_col=BACKGROUND_COL,
        background_filter=BACKGROUND_FILTER,
        spatial_key=SPATIAL_KEY,
        cell_coordinate_source=CELL_COORDINATE_SOURCE,
        cell_x_col=CELL_X_COL,
        cell_y_col=CELL_Y_COL,
        gene_names_col=GENE_NAMES_COL,
    )

    print(f"Running DenoIST for {sample_name}: {inp.counts.shape[0]} genes x {inp.counts.shape[1]} cells")
    result = denoist(
        inp,
        x_col="x",
        y_col="y",
        gene_col=GENE_COL,
        qv_col=QV_COL,
        qv_threshold=QV_THRESHOLD,
        distance=DISTANCE,
        nbins=NBINS,
        posterior_cutoff=POSTERIOR_CUTOFF,
        n_inits=N_INITS,
        max_iter=MAX_ITER,
        tol=TOL,
        backend=BACKEND,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        background_only=BACKGROUND_ONLY,
        include_self_twice=INCLUDE_SELF_TWICE,
        store_memberships=STORE_MEMBERSHIPS,
        store_posterior=STORE_POSTERIOR,
        random_state=RANDOM_STATE,
        progress=DENOIST_PROGRESS,
    )

    common_export = {
        "template_adata": inp.source_table,
        "copy_uns_from_template": COPY_UNS_FROM_TEMPLATE,
    }
    if OUTPUT_FORMAT == "spatialdata-zarr":
        output_path = sample_dir / f"{sample_name}.denoist.spatialdata.zarr"
        write_result_spatialdata_zarr(
            sdata,
            result,
            inp,
            output_path,
            table_key=OUTPUT_TABLE_KEY,
            overwrite=OVERWRITE,
            **common_export,
        )
    elif OUTPUT_FORMAT == "h5ad":
        output_path = sample_dir / f"{sample_name}.denoist.h5ad"
        write_result_h5ad(result, inp, output_path, overwrite=OVERWRITE, **common_export)
    elif OUTPUT_FORMAT == "anndata-zarr":
        output_path = sample_dir / f"{sample_name}.denoist.anndata.zarr"
        write_result_anndata_zarr(result, inp, output_path, overwrite=OVERWRITE, **common_export)
    else:
        raise ValueError(f"Unknown OUTPUT_FORMAT: {OUTPUT_FORMAT}")

    report = summarize_denoist_result(result, raw_counts=inp, top_n=TOP_N)
    write_report_csv(report, report_dir, overwrite=OVERWRITE)
    qc_plots = plot_qc(report, sample_name, qc_dir)

    run_metadata = {
        "sample": sample_name,
        "input_zarr": str(zarr_path),
        "output_path": str(output_path),
        "report_dir": str(report_dir),
        "qc_dir": str(qc_dir),
        "qc_plots": [str(p) for p in qc_plots],
        "n_genes": int(inp.counts.shape[0]),
        "n_cells": int(inp.counts.shape[1]),
        "raw_counts_total": report_metric(report, "raw_counts_total"),
        "adjusted_counts_total": report_metric(report, "adjusted_counts_total"),
        "removed_counts_total": report_metric(report, "removed_counts_total"),
        "removed_fraction": report_metric(report, "removed_fraction"),
        "mode": result.metadata.get("mode", "unknown"),
        "backend": result.metadata.get("backend"),
        "status": "ok",
    }
    with open(sample_dir / "run_metadata.json", "w", encoding="utf-8") as handle:
        json.dump(run_metadata, handle, indent=2)

    print(f"Finished {sample_name}: removed_fraction={run_metadata['removed_fraction']:.4g}")
    return run_metadata

## 5. Run The Folder

In [11]:
zarr_paths = discover_zarrs(INPUT_DIR, INPUT_GLOB, RECURSIVE)
if SAMPLE_LIMIT is not None:
    zarr_paths = zarr_paths[:SAMPLE_LIMIT]

print(f"Found {len(zarr_paths)} Zarr stores")
for path in zarr_paths:
    print(" -", path)

Found 12 Zarr stores
 - E:\cosmx\proseg_out\R6420_S1_TMA1_F00001_F00025.zarr
 - E:\cosmx\proseg_out\R6420_S1_TMA1_F00026_F00050.zarr
 - E:\cosmx\proseg_out\R6420_S1_TMA1_F00051_F00075.zarr
 - E:\cosmx\proseg_out\R6420_S1_TMA1_F00076_F00100.zarr
 - E:\cosmx\proseg_out\R6420_S1_TMA1_F00101_F00125.zarr
 - E:\cosmx\proseg_out\R6420_S1_TMA1_F00126_F00150.zarr
 - E:\cosmx\proseg_out\R6420_S2_TMA2_F00001_F00025.zarr
 - E:\cosmx\proseg_out\R6420_S2_TMA2_F00026_F00050.zarr
 - E:\cosmx\proseg_out\R6420_S2_TMA2_F00051_F00075.zarr
 - E:\cosmx\proseg_out\R6420_S2_TMA2_F00076_F00100.zarr
 - E:\cosmx\proseg_out\R6420_S2_TMA2_F00101_F00125.zarr
 - E:\cosmx\proseg_out\R6420_S2_TMA2_F00126_F00150.zarr


In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_rows = []

for zarr_path in zarr_paths:
    try:
        run_rows.append(run_one_zarr(zarr_path))
    except Exception as exc:
        error_row = {
            "sample": clean_name(zarr_path),
            "input_zarr": str(zarr_path),
            "status": "failed",
            "error": repr(exc),
        }
        run_rows.append(error_row)
        print(f"FAILED {zarr_path}: {exc}")
        traceback.print_exc()
        if not CONTINUE_ON_ERROR:
            raise

batch_summary = pd.DataFrame(run_rows)
batch_summary_path = OUTPUT_DIR / "batch_summary.csv"
batch_summary.to_csv(batch_summary_path, index=False)
batch_summary

Reading E:\cosmx\proseg_out\R6420_S1_TMA1_F00001_F00025.zarr


C:\Users\acheeseman\AppData\Local\Temp\ipykernel_18544\1630486866.py:8: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(zarr_path)


Running DenoIST for R6420_S1_TMA1_F00001_F00025: 18934 genes x 33743 cells
[denoistpy] Preparing input ...
[denoistpy] Preparing input done in 0.0s
[denoistpy] Computing local/background offsets ...
[denoistpy] Computing local/background offsets done in 17.4s
[denoistpy] Fitting Poisson mixture: 33743 cells in batches of 1024 ...


EM batches:   0%|          | 0/33 [00:00<?, ?batch/s]

KeyboardInterrupt: 

## 6. Batch QC

In [ ]:
ok = batch_summary.query("status == 'ok'").copy()
if ok.empty:
    print("No successful runs to plot.")
else:
    fig, ax = plt.subplots(figsize=(max(7, 0.35 * len(ok)), 4))
    ax.bar(ok["sample"], ok["removed_fraction"].astype(float), color="#4C78A8")
    ax.set_title("DenoIST removed fraction by sample")
    ax.set_xlabel("Sample")
    ax.set_ylabel("Removed fraction")
    ax.tick_params(axis="x", labelrotation=45)
    fig.tight_layout()
    out = OUTPUT_DIR / "batch_removed_fraction.png"
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"Wrote {out}")

## 7. Inspect An Output

For `spatialdata-zarr` outputs, corrected counts are stored in `tables[OUTPUT_TABLE_KEY]`. For `h5ad` or `anndata-zarr`, read the output as an AnnData object and inspect `X`, `layers["raw_counts"]`, `layers["denoist_corrected"]`, `obs`, `var`, and `uns["denoist"]`.

In [ ]:
if not ok.empty and OUTPUT_FORMAT == "spatialdata-zarr":
    example_path = Path(ok.iloc[0]["output_path"])
    example = sd.read_zarr(example_path)
    denoist_table = example.tables[OUTPUT_TABLE_KEY]
    display(denoist_table)
    display(denoist_table.uns["denoist"]["summary"])
elif not ok.empty:
    import anndata as ad

    example_path = Path(ok.iloc[0]["output_path"])
    if OUTPUT_FORMAT == "h5ad":
        denoist_table = ad.read_h5ad(example_path)
    else:
        denoist_table = ad.read_zarr(example_path)
    display(denoist_table)
    display(denoist_table.uns["denoist"]["summary"])